In [2]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from functools import reduce

In [3]:
path = '/Users/matheoledevehat/.fastai/data/human_numbers'

In [4]:
train_data = [l.strip() for l in open(path+"/train.txt").readlines()]
valid_data = [l.strip() for l in open(path+"/valid.txt").readlines()]

In [5]:
token_list = list(set(reduce(lambda x,y: x+" . "+y, train_data).split(' ')))

In [6]:
vocab = {token_list[i]:i for i in range(len(token_list))}

In [7]:
device = torch.device('mps')

In [8]:
seq_len = 16
vocab_size = len(vocab)
n_hidden = 64
n_layers = 2
p = 0.4
batch_size = 64
epochs = 100
alpha = 2
beta = 1
lr = 0.2
momentum = 0.9
wd = 1e-5 

In [9]:
class LM(torch.nn.Module):
    def __init__(self, vocab_size, n_hidden, n_layers, p_dropout):
        super().__init__()
        k = 1.0 / (2 * n_hidden) ** 0.5
        self.embedding = torch.nn.Parameter(torch.empty(vocab_size, n_hidden).uniform_(-0.1, 0.1))
        self.forgettings = torch.nn.Parameter(torch.empty(n_layers,2*n_hidden,n_hidden).uniform_(-k, k))
        self.input_gates = torch.nn.Parameter(torch.empty(n_layers,2*n_hidden,n_hidden).uniform_(-k, k))
        self.cell_gates = torch.nn.Parameter(torch.empty(n_layers,2*n_hidden,n_hidden).uniform_(-k, k))
        self.outputs = torch.nn.Parameter(torch.empty(n_layers,2*n_hidden, n_hidden).uniform_(-k, k))

        self.b_f = torch.nn.Parameter(torch.ones(n_layers, n_hidden))
        self.b_i = torch.nn.Parameter(torch.zeros(n_layers, n_hidden))
        self.b_c = torch.nn.Parameter(torch.zeros(n_layers, n_hidden))
        self.b_o = torch.nn.Parameter(torch.zeros(n_layers, n_hidden))

        
        self.cell_states = None
        self.hidden_states = None
        self.n_layers = n_layers

        self.p = 1-p_dropout
        self.bern = torch.distributions.bernoulli.Bernoulli(self.p)
        self.activations_before_dropout = []
        self.activations_after_dropout = []

    def forward(self, x):
        output = []
        self.activations_before_dropout = []
        self.activations_after_dropout = []
        if self.hidden_states == None or self.cell_states == None:
            self.cell_states = [torch.zeros(x.shape[0],n_hidden,device=device) for _ in range(n_layers)]
            self.hidden_states = [torch.zeros(x.shape[0],n_hidden,device=device) for _ in range(n_layers)]

        masks = None
        if self.training:
            masks = [self.bern.sample((x.shape[0], n_hidden)).to(device) / self.p for _ in range(self.n_layers + 1)] 
        
        for i in range(x.shape[1]):
            for j in range(self.n_layers):
                if j == 0:
                    x_t = self.embedding[x[:,i]]
                else:
                    x_t = self.hidden_states[j-1]

                if self.training:
                    #mask = self.bern.sample(x_t.shape).to(device)
                    x_t = x_t*masks[j]
                    
                input = torch.cat([self.hidden_states[j], x_t], dim=1)
                #input has a shape of batch_size*(2*n_hidden)
                forget_activation = F.sigmoid(input@self.forgettings[j]+self.b_f[j])
                input_activation = F.sigmoid(input@self.input_gates[j]+self.b_i[j])
                cell_activation = F.tanh(input@self.cell_gates[j]+self.b_c[j])
                output_activation = F.sigmoid(input@self.outputs[j]+self.b_o[j])
                self.cell_states[j] = self.cell_states[j]*forget_activation+input_activation*cell_activation
                self.hidden_states[j] = F.tanh(self.cell_states[j])*output_activation
                if j == self.n_layers-1:
                    final_out = self.hidden_states[j]
                    
                    if self.training:
                        self.activations_before_dropout.append(final_out)
                        #mask = self.bern.sample(final_out.shape).to(device)
                        final_out = final_out*masks[-1]
                        self.activations_after_dropout.append(final_out)
                    else:
                        self.activations_before_dropout.append(final_out)
                        self.activations_after_dropout.append(final_out)
                    
                    output.append(final_out@self.embedding.t())
        self.hidden_states = [s.detach() for s in self.hidden_states]
        self.cell_states = [s.detach() for s in self.cell_states]
        return torch.stack(output, dim=1)

    def reset(self):
        self.hidden_states = None
        self.cell_states = None

In [10]:
def loss_fn(model, pred, target, split):
    loss = torch.nn.CrossEntropyLoss()
    print(pred.shape, target.shape)
    loss_y = loss(pred.view(-1, vocab_size), target.view(-1))
    if split == "train":
        loss_y += alpha * torch.stack(model.activations_after_dropout).pow(2).mean()
        activations = torch.stack(model.activations_before_dropout)
        loss_y += beta * (activations[1:, :, :] - activations[:-1, :, :]).pow(2).mean()
    #loss_y += wd * (torch.stack(model.parameters())**2).sum()
    return loss_y

In [11]:
def encode(text):
    return [vocab[w] for w in text.split(' ')]

In [12]:
class NumberDataset(torch.utils.data.IterableDataset):
    def __init__(self, data, seq_len, batch_size):
        super().__init__()
        #raw_data = reduce(lambda x,y: x+y, [encode(line) for line in data])
        text_tensor = torch.tensor(data)
        
        n_tokens = len(text_tensor) - 1 
        tokens_per_stream = n_tokens // batch_size
        
        x_data = text_tensor[:batch_size * tokens_per_stream]
        y_data = text_tensor[1 : batch_size * tokens_per_stream + 1]
        
        x_data = x_data.view(batch_size, -1)
        y_data = y_data.view(batch_size, -1)
        
        self.batches = []
        
        for i in range(0, x_data.shape[1] - seq_len + 1, seq_len):
            x_chunk = x_data[:, i:i+seq_len]
            y_chunk = y_data[:, i:i+seq_len]
            
            if x_chunk.shape[1] == seq_len:
                self.batches.append((x_chunk, y_chunk))

    def __iter__(self):
        return iter(self.batches)

In [13]:
#ds = NumberDataset(train_data, seq_len, batch_size)
#valid_ds = NumberDataset(valid_data, seq_len, batch_size)

In [14]:
all_lines = [l.strip() for l in open(path+"/train.txt").readlines()] \
          + [l.strip() for l in open(path+"/valid.txt").readlines()]
text = ' . '.join(all_lines)
tokens = text.split(' ')
# build vocab from tokens here, including '.'
nums = [vocab[t] for t in tokens]

# then split 80/20 on the token stream itself, NOT on the files
cut = int(len(nums) * 0.8)
train_nums, valid_nums = nums[:cut], nums[cut:]
ds       = NumberDataset(train_nums, seq_len, batch_size)
valid_ds = NumberDataset(valid_nums, seq_len, batch_size)

In [15]:
model = LM(vocab_size, n_hidden, n_layers, p).to(device)

In [16]:
velocities = [torch.zeros_like(param) for param in model.parameters()]

In [17]:
for e in range(epochs):
    model.train()
    total_loss, n = 0.0, 0

    for x,y in ds:
        if x.shape == torch.Size([batch_size, seq_len]) and y.shape == torch.Size([batch_size, seq_len]):
            x = x.to(device)
            y = y.to(device)
            pred = model(x)
            loss = loss_fn(model, pred, y, "train")
            loss.backward()
            total_loss += loss.item() * x.size(0); n += x.size(0)

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    
            with torch.no_grad():
                for param, v in zip(model.parameters(), velocities):
                    g = param.grad + wd * param.data
                    v.mul_(momentum).add_(g)
                    param.data.add_(v, alpha=-lr)
                    param.grad.zero_()
    total_loss /= n
    model.reset()
    model.eval()
    with torch.no_grad():
        vloss = torch.zeros(1, device=device)
        l = 0
        for x,y in valid_ds:
            if x.shape == torch.Size([batch_size, seq_len]) and y.shape == torch.Size([batch_size, seq_len]):
                l+=1
                x = x.to(device)
                y = y.to(device)
                pred = model(x)
                vloss += loss_fn(model, pred, y, "test")
        vloss /= l
    model.reset()
    print(f"epoch {e+1}, training loss: {total_loss:.4f}, validation loss: {float(vloss.data):.4f}")

torch.Size([64, 16, 30]) torch.Size([64, 16])
torch.Size([64, 16, 30]) torch.Size([64, 16])
torch.Size([64, 16, 30]) torch.Size([64, 16])
torch.Size([64, 16, 30]) torch.Size([64, 16])
torch.Size([64, 16, 30]) torch.Size([64, 16])
torch.Size([64, 16, 30]) torch.Size([64, 16])
torch.Size([64, 16, 30]) torch.Size([64, 16])
torch.Size([64, 16, 30]) torch.Size([64, 16])
torch.Size([64, 16, 30]) torch.Size([64, 16])
torch.Size([64, 16, 30]) torch.Size([64, 16])
torch.Size([64, 16, 30]) torch.Size([64, 16])
torch.Size([64, 16, 30]) torch.Size([64, 16])
torch.Size([64, 16, 30]) torch.Size([64, 16])
torch.Size([64, 16, 30]) torch.Size([64, 16])
torch.Size([64, 16, 30]) torch.Size([64, 16])
torch.Size([64, 16, 30]) torch.Size([64, 16])
torch.Size([64, 16, 30]) torch.Size([64, 16])
torch.Size([64, 16, 30]) torch.Size([64, 16])
torch.Size([64, 16, 30]) torch.Size([64, 16])
torch.Size([64, 16, 30]) torch.Size([64, 16])
torch.Size([64, 16, 30]) torch.Size([64, 16])
torch.Size([64, 16, 30]) torch.Siz

KeyboardInterrupt: 

In [17]:
# no momentum, before dataset fix => epoch 60, training loss: 0.3856, validation loss: 1.1729
# no momentum, after dataset fix => epoch 60, training loss: 0.3887, validation loss: 0.8861
# momentum => epoch 60, training loss: 1.2058, validation loss: 1.5109
# momentum (dropout fix) => epoch 60, training loss: 0.3204, validation loss: 0.6554
# lr=0.2 => epoch 100, training loss: 0.2785, validation loss: 0.3608

In [18]:
# SGD: lr = 2, p = 0.4 => epoch 100, training loss: 0.0687, validation loss: 0.8437
# SGD: lr = 15,8,4,2 => epoch 100, training loss: 0.0602, validation loss: 0.6478
# SGD: lr = 15,4,2,1,0.1 p=0.4, wd = 1e-4 => epoch 100, training loss: 0.0925, validation loss: 0.8279
# AdamW: validation loss: >1
# Adam: validation loss: >1

In [20]:
# 1. Create a reverse dictionary to decode numbers back to words
idx_to_vocab = {v: k for k, v in vocab.items()}

def generate_text(model, prompt, n_words=15):
    model.eval()
    model.reset() # Start with a clean slate for the new sequence
    
    # 2. Encode the initial prompt (assumes space-separated words)
    tokens = [vocab[w] for w in prompt.strip().split(' ')]
    
    # 3. Pass the prompt through the model to build up the memory (hidden state)
    # Batch size is 1, sequence length is len(tokens)
    x = torch.tensor([tokens]).to(device)
    with torch.no_grad():
        out = model(x)
        
    # Grab the highest probability prediction for the VERY LAST word in the prompt
    next_token = out[0, -1, :].argmax().item()
    generated_tokens = [next_token]
    
    # 4. Generate new words one at a time (Auto-regressive loop)
    for _ in range(n_words - 1):
        # Pass just the newly generated token (seq_len = 1)
        # Because we don't call model.reset(), the hidden state flawlessly carries over!
        x = torch.tensor([[next_token]]).to(device)
        with torch.no_grad():
            out = model(x)
        
        next_token = out[0, -1, :].argmax().item()
        generated_tokens.append(next_token)
        
    # 5. Decode and format the final output
    generated_text = " ".join([idx_to_vocab[t] for t in generated_tokens])
    return f"{prompt} {generated_text}"

# Let's test it! (Make sure the prompt words exist in your vocab)
print(generate_text(model, "one two three", n_words=10))
print(generate_text(model, "eighty one eighty two", n_words=100))

one two three . four thousand four hundred four . four thousand four
eighty one eighty two . four thousand two hundred eighty three . four thousand two hundred eighty four . four thousand two hundred eighty five . four thousand two hundred eighty six . four thousand two hundred eighty seven . four thousand two hundred eighty eight . four thousand two hundred eighty nine . four thousand two hundred ninety . four thousand two hundred ninety one . four thousand two hundred ninety two . four thousand two hundred ninety three . four thousand two hundred ninety four . four thousand two hundred ninety five . four thousand two hundred ninety six . four thousand
